# Validation Croisée Temporelle : Deep Learning (LSTM)

Ce notebook propose une implémentation robuste de l'architecture **LSTM** pour les séries temporelles.

**Problématique résolue : Le Data Leakage**
Auparavant, le `MinMaxScaler` était appliqué sur l'ensemble complet avant la création des fenêtres temporelles, ce qui permettait au modèle de connaître implicitement les maximums et minimums de l'ensemble de test.

Ici, la normalisation est appliquée de façon **dynamique**, uniquement sur la fenêtre de données historiques courante.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import os

# Chargement
data_dir = '../data'
df = pd.read_csv(os.path.join(data_dir, 'merged_tourism_data_final.csv'))

# Préparation séquentielle (Sans Scaling Global)
def prepare_data_wf(df_in, window_size=12):
    df_p = df_in.copy()
    features = [c for c in ['Arrivals', 'Nights', 'is_covid', 'InterTourismeReceipts', 'REER', 'Oil_price', 'FDI', 'Poverty_rate'] if c in df_p.columns]
    df_p[features] = df_p[features].interpolate().bfill().ffill()
    data = df_p[features].values
    
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size, :])
        y.append(data[i+window_size, 0])
        
    return np.array(X), np.array(y)

window = 12
X_all, y_all = prepare_data_wf(df, window_size=window)
train_size = int(len(X_all) * 0.8)


## Entraînement Walk-Forward avec Scaling Dynamique
À chaque pas de temps, le modèle est instancié, et le scaler est ajusté uniquement sur l'historique.

In [ ]:
tscv = TimeSeriesSplit(n_splits=len(X_all) - train_size, test_size=1)
y_pred_dl = []
y_test_real_dl = []

print("Entraînement des LSTMs sur fenêtres glissantes...")

for train_index, test_index in tscv.split(X_all):
    if test_index[0] < train_size:
        continue
        
    X_train_wf, X_test_wf = X_all[train_index], X_all[test_index]
    y_train_wf, y_test_wf = y_all[train_index], y_all[test_index]
    
    # ----------------------------------------------------
    # ANTI-LEAKAGE : Scaling dynamique sur le Train Set courant
    # ----------------------------------------------------
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X_train_wf_flat = X_train_wf.reshape(-1, X_train_wf.shape[-1])
    X_train_wf_scaled = scaler_X.fit_transform(X_train_wf_flat).reshape(X_train_wf.shape)
    y_train_wf_scaled = scaler_y.fit_transform(y_train_wf.reshape(-1, 1))
    
    X_test_wf_flat = X_test_wf.reshape(-1, X_test_wf.shape[-1])
    X_test_wf_scaled = scaler_X.transform(X_test_wf_flat).reshape(X_test_wf.shape)
    
    # Architecture LSTM
    model = Sequential([
        LSTM(126, input_shape=(X_train_wf_scaled.shape[1], X_train_wf_scaled.shape[2])),
        Dropout(0.16),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.002), loss='mse')
    
    # Entraînement
    model.fit(X_train_wf_scaled, y_train_wf_scaled, epochs=5, batch_size=16, verbose=0)
    
    # Prédiction
    pred_scaled = model.predict(X_test_wf_scaled, verbose=0)
    pred = scaler_y.inverse_transform(pred_scaled)
    
    y_pred_dl.append(pred[0, 0])
    y_test_real_dl.append(y_test_wf[0])

# Évaluation
metrics = {
    'R2': r2_score(y_test_real_dl, y_pred_dl),
    'RMSE': np.sqrt(mean_squared_error(y_test_real_dl, y_pred_dl)),
    'MAE': mean_absolute_error(y_test_real_dl, y_pred_dl),
    'MAPE': mean_absolute_percentage_error(y_test_real_dl, y_pred_dl)
}
print(f"\nMétriques Finales LSTM Walk-Forward :\n{metrics}")

plt.figure(figsize=(10,5))
plt.plot(y_test_real_dl, label='Réel', color='black')
plt.plot(y_pred_dl, label='Prédiction LSTM', color='blue', linestyle='--')
plt.title('LSTM - Réel vs Prédiction (Walk-Forward)')
plt.legend()
plt.grid()
plt.show()
